In [1]:
!git clone https://github.com/gaomengli7/OntoGene.git


Cloning into 'OntoGene'...
remote: Enumerating objects: 186, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 186 (delta 28), reused 179 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (186/186), 300.34 KiB | 11.12 MiB/s, done.
Resolving deltas: 100% (28/28), done.
Filtering content: 100% (37/37), 1.05 GiB | 62.33 MiB/s, done.


In [2]:
import os

model_path = "OntoGene/data/output_data/model"

print("Does model directory exist?", os.path.exists(model_path))

if os.path.exists(model_path):
    for root, dirs, files in os.walk(model_path):
        print("\nDirectory:", root)
        if files:
            print("Files:", files[:5])  # show first few files only


Does model directory exist? True

Directory: OntoGene/data/output_data/model

Directory: OntoGene/data/output_data/model/promotercore

Directory: OntoGene/data/output_data/model/promotercore/OntoGeneModel
Files: ['tokenizer_config.json', 'special_tokens_map.json', 'config.json', 'pytorch_model.bin', 'vocab.txt']


In [3]:
from pathlib import Path

fasta_path = Path("Human.fasta")

print("Human.fasta exists:", fasta_path.exists())

# Simple FASTA reader
def read_fasta(file_path):
    sequences = {}
    with open(file_path, "r") as f:
        seq_id = None
        seq = []
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if seq_id:
                    sequences[seq_id] = "".join(seq)
                seq_id = line[1:]
                seq = []
            else:
                seq.append(line.upper())
        if seq_id:
            sequences[seq_id] = "".join(seq)
    return sequences

human_sequences = read_fasta(fasta_path)

print("Number of sequences:", len(human_sequences))

# Show first 3 sequences (ID + length only)
for i, (k, v) in enumerate(human_sequences.items()):
    print(k, "length:", len(v))
    if i == 2:
        break


Human.fasta exists: True
Number of sequences: 48342
FP020532 UNC119_2            :+U  EU:NC; range  -249 to    50. length: 300
FP020301 CNTROB_1            :+U  EU:NC; range  -249 to    50. length: 300
FP008839 FAM135A_1           :+U  EU:NC; range  -249 to    50. length: 300


In [4]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# Path to pretrained OntoGene model
MODEL_PATH = "OntoGene/data/output_data/model/promotercore/OntoGeneModel"

print("Loading tokenizer...")
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)

print("Loading model...")
model = BertForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()  # VERY IMPORTANT: inference mode

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded on:", device)


Loading tokenizer...
Loading model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at OntoGene/data/output_data/model/promotercore/OntoGeneModel and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded on: cuda


In [5]:
# Take one sequence only (sanity check)
sample_id, sample_seq = next(iter(human_sequences.items()))

print("Sequence ID:", sample_id)
print("Sequence length:", len(sample_seq))

# Tokenize
inputs = tokenizer(
    sample_seq,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=300
)

# Move to device
inputs = {k: v.to(device) for k, v in inputs.items()}

# Inference
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)

print("Logits:", logits.cpu().numpy())
print("Probabilities:", probs.cpu().numpy())


Sequence ID: FP020532 UNC119_2            :+U  EU:NC; range  -249 to    50.
Sequence length: 300
Logits: [[-0.30932146  0.23258924]]
Probabilities: [[0.3677432 0.6322568]]


In [6]:
from tqdm import tqdm
import pandas as pd

BATCH_SIZE = 32  # safe for CPU; increase to 64/128 if GPU

results = []

sequence_items = list(human_sequences.items())

for i in tqdm(range(0, len(sequence_items), BATCH_SIZE)):
    batch_items = sequence_items[i:i+BATCH_SIZE]
    batch_seqs = [seq for _, seq in batch_items]
    batch_ids = [sid for sid, _ in batch_items]

    inputs = tokenizer(
        batch_seqs,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=300
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()

    for sid, p in zip(batch_ids, probs):
        results.append({
            "sequence_id": sid,
            "non_promoter_prob": float(p[0]),
            "promoter_prob": float(p[1])
        })


100%|██████████| 1511/1511 [12:38<00:00,  1.99it/s]


In [7]:
import pandas as pd

df = pd.DataFrame(results)

output_file = "OntoGene_Human_independent_predictions.csv"
df.to_csv(output_file, index=False)

print("✅ Saved file:", output_file)
print("\nPreview:")
print(df.head())


✅ Saved file: OntoGene_Human_independent_predictions.csv

Preview:
                                         sequence_id  non_promoter_prob  \
0  FP020532 UNC119_2            :+U  EU:NC; range...           0.367743   
1  FP020301 CNTROB_1            :+U  EU:NC; range...           0.367743   
2  FP008839 FAM135A_1           :+U  EU:NC; range...           0.367743   
3  FP002661 VSNL1_3             :+U  EU:NC; range...           0.367743   
4  FP004964 SLMAP_1             :+U  EU:NC; range...           0.367743   

   promoter_prob  
0       0.632257  
1       0.632257  
2       0.632257  
3       0.632257  
4       0.632257  
